# RealBasicVSR — Drive Frame Saver & Resume Master

এই Notebook-এর কাজ:
1. Google Drive মাউন্ট করা
2. RealBasicVSR-এর proven environment তৈরি করা
3. Drive-এর একটি project folder থেকে input video নেওয়া
4. প্রতিটি AI-upscaled frame সরাসরি Google Drive-এ PNG হিসেবে save করা
5. মাঝপথে runtime/ইন্টারনেট বন্ধ হলে আবার চালালে আগে তৈরি হওয়া frame **skip** করা
6. সব 485/যত frame আছে সম্পূর্ণ হলে এখানেই থামা

**গুরুত্বপূর্ণ:** Runtime বন্ধ হলে `/content` হারাতে পারে, কিন্তু Google Drive-এর frame থাকবে।

In [ ]:
# STEP 1 — Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# STEP 2 — Project folder
# চাইলে শুধু PROJECT নামটি পরিবর্তন করুন।
PROJECT = "RealBasicVSR_Project"
DRIVE_ROOT = f"/content/drive/MyDrive/{PROJECT}"
INPUT_DIR = f"{DRIVE_ROOT}/input"
FRAMES_DIR = f"{DRIVE_ROOT}/frames"

import os
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(FRAMES_DIR, exist_ok=True)

print("Project:", DRIVE_ROOT)
print("Input :", INPUT_DIR)
print("Frames:", FRAMES_DIR)

## STEP 3 — Input video

প্রথমবার আপনার input MP4-টি Google Drive-এর এই folder-এ রাখুন:

`My Drive / RealBasicVSR_Project / input`

তারপর নিচের cell চালান।

In [ ]:
# STEP 3 — Show input files
!ls -lh "$INPUT_DIR" || true

In [ ]:
# STEP 4 — Install the exact proven environment
!sudo apt-get update -qq
!sudo apt-get install -y python3.10 python3.10-venv
!rm -rf /content/rbvsr_legacy
!python3.10 -m venv /content/rbvsr_legacy

!/content/rbvsr_legacy/bin/python -m pip install --upgrade pip==22.0.2
!/content/rbvsr_legacy/bin/python -m pip install numpy==1.26.4
!/content/rbvsr_legacy/bin/python -m pip install torch==1.13.1+cu117 torchvision==0.14.1 --extra-index-url https://download.pytorch.org/whl/cu117
!/content/rbvsr_legacy/bin/python -m pip install openmim==0.3.9
!/content/rbvsr_legacy/bin/python -m pip install mmcv-full==1.7.2
!/content/rbvsr_legacy/bin/python -m pip install numpy==1.26.4 opencv-python==4.8.1.78
!/content/rbvsr_legacy/bin/python -m pip install mmedit==0.16.1 tqdm==4.65.2

In [ ]:
# STEP 5 — Clone repo + config + checkpoint
!rm -rf /content/RealBasicVSR
!git clone -q https://github.com/ckkelvinchan/RealBasicVSR.git /content/RealBasicVSR
!mkdir -p /content/RealBasicVSR/configs /content/RealBasicVSR/checkpoints
!wget -q https://raw.githubusercontent.com/ckkelvinchan/RealBasicVSR/master/configs/realbasicvsr_x4.py -O /content/RealBasicVSR/configs/realbasicvsr_x4.py
!pip install -q gdown
!gdown 1OYR1J2GXE90Zu2gVU5xc0t0P_UmKH7ID -O /content/RealBasicVSR/checkpoints/RealBasicVSR.pth
!ls -lh /content/RealBasicVSR/checkpoints/RealBasicVSR.pth

In [ ]:
# STEP 6 — Verify GPU + exact package versions
!/content/rbvsr_legacy/bin/python - <<'PY'
import torch, torchvision, numpy, cv2, mmcv, mmedit
print("Python:", __import__("sys").version.split()[0])
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("Torchvision:", torchvision.__version__)
print("NumPy:", numpy.__version__)
print("MMCV:", mmcv.__version__)
print("MMEditing:", mmedit.__version__)
print("OpenCV:", cv2.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
PY

In [ ]:
# STEP 7 — Find the input video and show its basic properties
import os, glob, subprocess
videos = sorted(glob.glob(INPUT_DIR + "/*.mp4") + glob.glob(INPUT_DIR + "/*.MP4"))
if not videos:
    raise FileNotFoundError("INPUT folder-এ MP4 পাওয়া যায়নি। আগে ভিডিওটি Google Drive-এর input folder-এ রাখুন।")
INPUT_VIDEO = videos[0]
print("Using:", INPUT_VIDEO)
!ffprobe -v error -select_streams v:0 -show_entries stream=codec_name,width,height,r_frame_rate,duration -of default=noprint_wrappers=1 "$INPUT_VIDEO"

In [ ]:
# STEP 8 — Create the Drive-resumable inference script
script = r'''
import os, cv2, torch
from tqdm import tqdm
from mmcv import Config
from mmedit.models import build_model

INPUT_VIDEO = os.environ["RBVSR_INPUT"]
OUT_DIR = os.environ["RBVSR_FRAMES"]
CFG = "/content/RealBasicVSR/configs/realbasicvsr_x4.py"
CKPT = "/content/RealBasicVSR/checkpoints/RealBasicVSR.pth"

os.makedirs(OUT_DIR, exist_ok=True)

cfg = Config.fromfile(CFG)
model = build_model(cfg.model, train_cfg=None, test_cfg=cfg.get("test_cfg"))
model = model.cuda().eval()
checkpoint = torch.load(CKPT, map_location="cpu")
state_dict = checkpoint.get("state_dict", checkpoint)
model.load_state_dict(state_dict, strict=False)

cap = cv2.VideoCapture(INPUT_VIDEO)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)
print(f"Total frames: {total}, FPS: {fps}")

# The proven workflow performs one-frame inference at a time to fit a T4.
for i in tqdm(range(total), desc="Drive-safe upscaling"):
    out_path = os.path.join(OUT_DIR, f"{i+1:05d}.png")
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1000:
        continue

    cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ok, frame = cap.read()
    if not ok:
        raise RuntimeError(f"Could not read frame {i+1}")

    # BGR uint8 -> RGB float tensor, NCHW
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = torch.from_numpy(rgb).float().permute(2,0,1).unsqueeze(0).unsqueeze(0).cuda() / 255.0

    with torch.no_grad():
        out = model(img, test_mode=True)
    result = out["output"] if isinstance(out, dict) else out
    if result.dim() == 5:
        result = result[0,0]
    elif result.dim() == 4:
        result = result[0]
    result = result.clamp(0,1).permute(1,2,0).cpu().numpy()
    bgr = cv2.cvtColor((result*255).round().astype("uint8"), cv2.COLOR_RGB2BGR)
    cv2.imwrite(out_path, bgr)

cap.release()
print("DONE:", total, "frames. Drive folder:", OUT_DIR)
'''
open("/content/RealBasicVSR/inference_drive_resume.py","w").write(script)
print("Script created.")

In [ ]:
# STEP 9 — Run. Existing Drive frames are automatically skipped.
import os, glob
os.environ["RBVSR_INPUT"] = INPUT_VIDEO
os.environ["RBVSR_FRAMES"] = FRAMES_DIR
!RBVSR_INPUT="$RBVSR_INPUT" RBVSR_FRAMES="$RBVSR_FRAMES" /content/rbvsr_legacy/bin/python /content/RealBasicVSR/inference_drive_resume.py

In [ ]:
# STEP 10 — Verify Drive frame count
import glob, os
frames = sorted(glob.glob(FRAMES_DIR + "/*.png"))
print("Frames on Google Drive:", len(frames))
print("First:", os.path.basename(frames[0]) if frames else "none")
print("Last :", os.path.basename(frames[-1]) if frames else "none")

### যদি মাঝপথে runtime বন্ধ হয়ে যায়

নতুন Colab/runtime-এ আবার এই Notebook চালান। **একই Drive project folder** ব্যবহার করুন।

নতুন করে setup হওয়ার পর Step 9 চালালে:
- Drive-এ থাকা `00001.png`, `00002.png` ... যেগুলো আছে সেগুলো আবার তৈরি করবে না।
- শুধু missing frame-গুলো তৈরি করবে।

সব frame শেষ হলে দ্বিতীয় Master Notebook দিয়ে ভিডিও বানানো যাবে।